In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv") \
    .option("inferSchema", True) \
    .option("header", True) \
    .load("/Volumes/workspace/default/pyspark_lab_1_csv/BigMart Sales.csv")

## Split, Indexing & Explode
- **Split** - Splits a string into an array object
- **Explode** - used to convert an array or map object into multiple rows

In [0]:
df.display()

In [0]:
df1 = df.withColumn("Item_Type", split(col("Item_Type"),' '))

In [0]:
df1.display()

In [0]:
df2 = df.withColumn("Item_Type", split(col("Item_Type"),' ')[0])

In [0]:
df2.display()

In [0]:
df3 = df1.withColumn("new_item_type", explode(col("Item_Type")))
df3.display()

### Array_Contains Method

In [0]:
df2 = df1.withColumn("Type_Flag", array_contains(col("Item_Type"), "Snack"))

In [0]:
df2.display()

## GROUP BY operation

In [0]:
df1 = df.groupBy("Item_Type").agg(sum("Item_MRP").alias("Total_Sum"))

In [0]:
df1.display()

In [0]:
df2 = df.groupBy("Item_Type", "Item_Fat_Content").agg(sum("Item_MRP").alias("Total_Sum"))

In [0]:
df2.display()

In [0]:
df3 = df.groupBy(col("Item_Type"), col("Outlet_Size")).agg(sum("Item_MRP"), avg("Item_Outlet_Sales"))

In [0]:
df3.display()

## collect_list() Method

In [0]:
data = [('Aditya', 'book1'),
        ('Sam', 'book2'),
        ('Asha', 'book1'),
        ('Sam', 'book1'),
        ('Aditya', 'book2')]
schema = "name string, book string"
df = spark.createDataFrame(data=data, schema=schema)
df.display()

In [0]:
df_collect_list = df.groupBy("name").agg(collect_list("book"))

In [0]:
df_collect_list.display()

## When Otherwise Function

In [0]:
df1 = df.withColumn("Veg/NonVeg", when(col("Item_Type") == "Meat", "Non-Veg").otherwise("Veg"))

In [0]:
df1.display()

In [0]:
df2 = df1.withColumn("Exp_Flag", when((col("Veg/NonVeg") == "Veg") & (col("Item_MRP") > 100), "Veg_Expensive")\
.when((col("Veg/NonVeg") == "Non-Veg") & (col("Item_MRP") > 100), "Non-Veg_Expensive")\
.when((col("Veg/NonVeg") == "Veg") & (col("Item_MRP") <= 100), "Veg_LessExpensive")\
.when((col("Veg/NonVeg") == "Non-Veg") & (col("Item_MRP") <= 100), "Non-Veg_LessExpensive")\
.otherwise("N/A"))

In [0]:
df2.display()

## Window Functions
### Row_Number()
### Rank()
### Dense_Rank()
### Cumulative_Sum()

In [0]:
from pyspark.sql.window import Window

In [0]:
df1 = df.withColumn("Row_Num_Col", row_number().over(Window.orderBy(col("Item_Identifier"))))

In [0]:
df1.display()

In [0]:
df2 = df.withColumn("Ranked_Column",rank().over(Window.orderBy(col("Item_Identifier"))))
df2.display()

In [0]:
df3 = df.withColumn("Dense_Ranked_Column",rank().over(Window.orderBy(col("Item_Identifier"))))
df3.display()

In [0]:
df4 = df.withColumn("Ranked_Column2", dense_rank().over(Window.partitionBy("Outlet_Type").orderBy("Item_Identifier", "Item_Outlet_Sales")))

In [0]:
df4.display()

In [0]:
df5 = df.withColumn("Cum_Sum", sum("Item_MRP").over(Window.orderBy("Item_Type").rowsBetween(Window.unboundedPreceding, Window.currentRow)))

In [0]:
df5.display()